In [0]:
from pyspark.sql.functions import col, current_date, when, explode_outer, lit
from delta.tables import DeltaTable

# ========== FUNCTION 1: FLATTEN ALL DATA ==========
def flatten_all_data(df):
    df_exploded = df.withColumn(
        "previous_company_names_exploded",
        explode_outer("previous_company_names")
    )

    return df_exploded.select(
        "company_name",
        "company_number",
        "company_status",
        col("accounts.accounting_reference_date.day").alias("acc_ref_day"),
        col("accounts.accounting_reference_date.month").alias("acc_ref_month"),
        col("accounts.last_accounts.made_up_to").alias("last_made_up_to"),
        col("accounts.last_accounts.period_end_on").alias("last_period_end"),
        col("accounts.last_accounts.period_start_on").alias("last_period_start"),
        col("accounts.last_accounts.type").alias("last_accounts_type"),
        col("accounts.next_accounts.due_on").alias("next_due_on"),
        col("accounts.next_accounts.overdue").alias("next_overdue"),
        col("accounts.next_accounts.period_end_on").alias("next_period_end"),
        col("accounts.next_accounts.period_start_on").alias("next_period_start"),
        col("accounts.next_due").alias("next_due"),
        col("accounts.next_made_up_to").alias("next_made_up_to"),
        col("accounts.overdue").alias("accounts_overdue"),
        col("previous_company_names_exploded.name").alias("previous_company_name"),
        col("previous_company_names_exploded.effective_from").alias("effective_from"),
        col("previous_company_names_exploded.ceased_on").alias("ceased_on"),
        when(col("previous_company_names_exploded.name").isNotNull(), "previous_names")
        .otherwise("accounts").alias("record_type")
    )

# ========== FUNCTION 2: SCD2 MERGE ==========
def scd2_merge(spark, source_df, target_table, business_key):

    # Remove duplicate records before merge
    source_df = source_df.dropDuplicates([
        "company_number",
        "record_type",
        "previous_company_name",
        "effective_from",
        "ceased_on"
    ])

    source_df = source_df.withColumn("effective_start_date", current_date()) \
                         .withColumn("effective_end_date", lit(None).cast("date")) \
                         .withColumn("is_current", lit(1))

    if not spark.catalog.tableExists(target_table):
        print(f"Creating table: {target_table}")

        source_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        print("✓ Table created")

    else:
        print(f"Merging into table: {target_table}")

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"t.{k} = s.{k}" for k in business_key]
        )

        delta_table.alias("t").merge(
            source_df.alias("s"),
            merge_condition + " AND t.is_current = 1"
        ).whenMatchedUpdate(
            set={
                "is_current": "0",
                "effective_end_date": "current_date()"
            }
        ).execute()

        source_df.write.format("delta") \
            .mode("append") \
            .saveAsTable(target_table)

        print("✓ Merge completed")

# ========== MAIN EXECUTION ==========

print("\n[STEP 1] Reading Bronze table...")
df = spark.read.table("corporate_data_lakehouse.bronze.comp_house_overview")
print(f"✓ Loaded {df.count()} records")

print("\n[STEP 2] Flattening data (accounts + previous names)...")
df_final = flatten_all_data(df)

# Remove duplicates after flattening
df_final = df_final.dropDuplicates([
    "company_number",
    "record_type",
    "previous_company_name",
    "effective_from",
    "ceased_on"
])

print(f"✓ Flattened {df_final.count()} records after deduplication")

print("\n[STEP 3] Merging into Silver layer...")
scd2_merge(
    spark,
    df_final,
    "corporate_data_lakehouse.silver.comp_overview",
    ["company_number", "record_type"]
)

print("\n[STEP 4] Verifying results...")
result = spark.table("corporate_data_lakehouse.silver.comp_overview")

print(f"✓ Total records: {result.count()}")
print(f"✓ Total columns: {len(result.columns)}")

print("✅ SCD2 TABLE CREATED SUCCESSFULLY!")